### Feature Engineering | Demandes d’asile UNHCR & IDMC

### Objectif

Transformer le dataset analysé dans l’EDA en un **Feature Set reproductible, traçable et compatible avec un scoring réel à T=0**.

### Principes

- aucune suppression silencieuse de lignes métier ;
- `lag1 = T-1` et `lag2 = T-2` exactement ;
- aucune variable future dans `X` ;
- les colonnes techniques, redondantes ou trop incomplètes sont retirées du dataset ML, mais conservées dans la source ;
- les variables de décisions et IDMC sont privilégiées sous forme retardée ;
- un journal explique pourquoi chaque colonne est conservée ou exclue.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [2]:
DATA_PATH = Path("../data/consolidated/dataset_demandes_consolide_detail.csv")
OUTPUT_DIR = Path("../data/feature_engineering_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_source = pd.read_csv(DATA_PATH, low_memory=False)
df = df_source.copy()

print("Dataset source :", df.shape)
print("Période :", int(df["year"].min()), "→", int(df["year"].max()))


Dataset source : (120597, 50)
Période : 2000 → 2025


### 1. Définition du grain métier

La cible est construite au niveau du même segment :

`coo_id × coa_id × procedure_type × app_type × dec_level × app_pc`

Le modèle cherchera à anticiper si ce même segment connaîtra une hausse supérieure à 15 % l’année suivante.


In [3]:
SEGMENT = [
    "coo_id",
    "coa_id",
    "procedure_type",
    "app_type",
    "dec_level",
    "app_pc"
]

TARGET = "hausse_critique_demandes_t1"

missing_segment_cols = [c for c in SEGMENT if c not in df.columns]
assert not missing_segment_cols, f"Colonnes du grain manquantes : {missing_segment_cols}"

n_duplicates = df.duplicated(SEGMENT + ["year"]).sum()
print("Doublons au grain segment × année :", int(n_duplicates))

if n_duplicates > 0:
    print("ATTENTION : doublons à auditer. Aucune agrégation automatique n'est appliquée.")


Doublons au grain segment × année : 0


### 2. Construction stricte des lags temporels

Un simple `shift(1)` peut être trompeur lorsqu’une année est absente.

Ici :
- `applied_lag1` = valeur du même segment à T−1 ;
- `applied_lag2` = valeur du même segment à T−2.

Ainsi, si 2020 manque entre 2019 et 2021, 2019 ne devient pas artificiellement le lag1 de 2021.


*Un lag correspond à une valeur passée.*

In [9]:
def add_exact_lag(base_df, columns, lag, segment_cols):
    right_cols = segment_cols + ["year"] + columns
    right = base_df[right_cols].copy()
    right["year"] = right["year"] + lag
    right = right.rename(columns={c: f"{c}_lag{lag}" for c in columns})

    return base_df.merge(
        right,
        on=segment_cols + ["year"],
        how="left",
        validate="one_to_one"
    )

df_fe = add_exact_lag(
    df,
    columns=["applied"],
    lag=1,
    segment_cols=SEGMENT
)

df_fe = add_exact_lag(
    df_fe,
    columns=["applied"],
    lag=2,
    segment_cols=SEGMENT
)

display(
    df_fe[
        SEGMENT + ["year", "applied", "applied_lag1", "applied_lag2"]
    ].head(15)
)


,coo_id,coa_id,procedure_type,app_type,dec_level,app_pc,year,applied,applied_lag1,applied_lag2
0,2,11,G,A,AR,C,2006,14,NaN,NaN
1,3,11,G,A,AR,C,2006,21,NaN,NaN
2,4,11,G,A,AR,C,2006,5,NaN,NaN
3,8,11,G,A,AR,C,2006,38,NaN,NaN
4,14,11,G,A,AR,C,2006,11,NaN,NaN
5,20,11,G,A,AR,C,2006,211,NaN,NaN
6,24,11,G,A,AR,C,2006,5,NaN,NaN
7,27,11,G,A,AR,C,2006,5,NaN,NaN
8,30,11,G,A,AR,C,2006,5,NaN,NaN
9,32,11,G,A,AR,C,2006,8,NaN,NaN


### 3. Features de dynamique passée


Ces variables décrivent la dynamique récente du segment à partir d’informations disponibles à T et dans le passé.

L'objectif est de ne plus seulement dire au modèle combien de demandes existaient dans le passé, mais de lui décrire la trajectoire récente du segment.

`Les features créées:`
- `absolute_change_past_1` mesure le nombre de demandes supplémentaires ou en moins entre T−1 et T: 150 - 120 = +30
- `growth_past_1` exprime la même évolution, mais en pourcentage : 150-120/120 = 25%
- `growth_past_2` regarde la période précédente, entre T−2 et T−1 : 120-100/100=20%
- `two_consecutive_increases`résume les informations précedentes, si les obervations augmentent ou diminuent.
- `applied_mean_3y` calcule le volume moyen sur T, T−1 et T−2
- `applied_std_3y` mesure la variabilité des volumes sur ces trois années

In [10]:
df_fe["absolute_change_past_1"] = (
    df_fe["applied"] - df_fe["applied_lag1"]
)

df_fe["absolute_change_past_2"] = (
    df_fe["applied_lag1"] - df_fe["applied_lag2"]
)

df_fe["growth_past_1"] = np.where(
    df_fe["applied_lag1"].gt(0),
    (df_fe["applied"] - df_fe["applied_lag1"])
    / df_fe["applied_lag1"],
    np.nan
)

df_fe["growth_past_2"] = np.where(
    df_fe["applied_lag2"].gt(0),
    (df_fe["applied_lag1"] - df_fe["applied_lag2"])
    / df_fe["applied_lag2"],
    np.nan
)

has_two_growths = (
    df_fe["growth_past_1"].notna()
    & df_fe["growth_past_2"].notna()
)

df_fe["two_consecutive_increases"] = pd.Series(
    pd.NA,
    index=df_fe.index,
    dtype="Int8"
)

df_fe.loc[has_two_growths, "two_consecutive_increases"] = (
    (df_fe.loc[has_two_growths, "growth_past_1"] > 0)
    & (df_fe.loc[has_two_growths, "growth_past_2"] > 0)
).astype("Int8")

history_3 = df_fe[["applied", "applied_lag1", "applied_lag2"]]
df_fe["applied_mean_3y"] = history_3.mean(axis=1, skipna=False)
df_fe["applied_std_3y"] = history_3.std(axis=1, skipna=False)

display(
    df_fe[
        ["year", *SEGMENT, "applied",
         "applied_lag1", "applied_lag2",
         "growth_past_1", "growth_past_2",
         "absolute_change_past_1",
         "two_consecutive_increases",
         "applied_mean_3y", "applied_std_3y"]
    ].head(15)
)


,year,coo_id,coa_id,procedure_type,app_type,dec_level,app_pc,applied,applied_lag1,applied_lag2,growth_past_1,growth_past_2,absolute_change_past_1,two_consecutive_increases,applied_mean_3y,applied_std_3y
0,2006,2,11,G,A,AR,C,14,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN
1,2006,3,11,G,A,AR,C,21,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN
2,2006,4,11,G,A,AR,C,5,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN
3,2006,8,11,G,A,AR,C,38,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN
4,2006,14,11,G,A,AR,C,11,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN
5,2006,20,11,G,A,AR,C,211,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN
6,2006,24,11,G,A,AR,C,5,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN
7,2006,27,11,G,A,AR,C,5,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN
8,2006,30,11,G,A,AR,C,5,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN
9,2006,32,11,G,A,AR,C,8,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN


Les variables temporelles permettent d’identifier si une hausse des demandes est ponctuelle ou si elle se répète sur plusieurs années. Lorsqu’une année T−1 ou T−2 est absente, la valeur du lag reste manquante (NaN) : aucune valeur n’est inventée ou remplacée arbitrairement. Cela garantit que les évolutions calculées reposent uniquement sur un historique réellement disponible.


### 4. Décisions et IDMC : stratégie anti-leakage

La disponibilité des données de décisions et IDMC à T peut dépendre du calendrier de publication. Pour une V1 conservatrice, on privilégie donc des versions **T−1**.

Cette étape sert à éviter que le modèle utilise, pendant son entraînement, une information qu’il n’aurait pas encore connue au moment où il doit faire sa prédiction.


In [13]:
conditional_context_cols = [
    "decisions_recognized",
    "decisions_other",
    "decisions_rejected",
    "decisions_closed",
    "decisions_total",
    "recognition_rate",
    "rejection_rate",
    "idmc_total_origin",
    "has_decisions_data",
    "has_idmc_data"
]

conditional_context_cols = [
    c for c in conditional_context_cols
    if c in df.columns
]

context_right = df[SEGMENT + ["year"] + conditional_context_cols].copy()
context_right["year"] = context_right["year"] + 1
context_right = context_right.rename(
    columns={c: f"{c}_lag1" for c in conditional_context_cols}
)

df_fe = df_fe.merge(
    context_right,
    on=SEGMENT + ["year"],
    how="left",
    validate="one_to_one"
)

valid_decision_total_lag1 = df_fe["decisions_total_lag1"].gt(0)

df_fe["refugee_recognition_rate_lag1"] = np.where(
    valid_decision_total_lag1,
    df_fe["decisions_recognized_lag1"] / df_fe["decisions_total_lag1"],
    np.nan
)

df_fe["protection_rate_lag1"] = np.where(
    valid_decision_total_lag1,
    (
        df_fe["decisions_recognized_lag1"]
        + df_fe["decisions_other_lag1"]
    ) / df_fe["decisions_total_lag1"],
    np.nan
)

print("Variables contextuelles retardées créées.")


Variables contextuelles retardées créées.


### 5. Construction stricte de la cible T+1

La cible vaut :
- `1` si la hausse de `applied` est > 15 % à T+1 ;
- `0` si elle est ≤ 15 % ;
- `NA` si le même segment n’existe pas réellement à T+1.


In [14]:
future = df[SEGMENT + ["year", "applied"]].copy()
future["year"] = future["year"] - 1
future = future.rename(columns={"applied": "applied_t1"})

df_fe = df_fe.merge(
    future,
    on=SEGMENT + ["year"],
    how="left",
    validate="one_to_one"
)

df_fe["year_t1"] = np.where(
    df_fe["applied_t1"].notna(),
    df_fe["year"] + 1,
    np.nan
)

df_fe["growth_t1"] = np.where(
    df_fe["applied"].gt(0) & df_fe["applied_t1"].notna(),
    (df_fe["applied_t1"] - df_fe["applied"]) / df_fe["applied"],
    np.nan
)

df_fe[TARGET] = pd.Series(pd.NA, index=df_fe.index, dtype="Int8")
eligible = df_fe["growth_t1"].notna()

df_fe.loc[eligible, TARGET] = (
    df_fe.loc[eligible, "growth_t1"] > 0.15
).astype("Int8")

print("Lignes totales :", len(df_fe))
print("Lignes labellisables :", int(eligible.sum()))
print("Lignes sans cible T+1 :", int((~eligible).sum()))
print("Taux de labellisation :", f"{eligible.mean():.2%}")


Lignes totales : 120597
Lignes labellisables : 80032
Lignes sans cible T+1 : 40565
Taux de labellisation : 66.36%


**Éligibilité à la modélisation :** Sur les `120 597 observations, 80 032 (66,36 %)` disposent d’une observation du même segment à T+1 et peuvent donc recevoir une cible fiable. Les `40 565 observations restantes (33,64 %)` ne sont pas utilisées pour l’apprentissage supervisé car leur évolution à T+1 n’est pas observable. Elles sont néanmoins conservées séparément pour assurer la traçabilité et ne sont pas assimilées artificiellement à la classe 0.

### 6. Décision Feature Set

Les variables temporelles (`applied_lag1`, `applied_lag2`, `growth_past_1`, `growth_past_2`, `two_consecutive_increases`) sont retenues comme **features candidates** afin de représenter l’historique et la persistance des évolutions. Les variables de solutions sont écartées de la V1 en raison de leur forte incomplétude, tandis que les identifiants techniques et toutes les variables futures (`year_t1`, `applied_t1`, `growth_t1`, cible) sont exclues afin d’éviter toute **fuite de cible**.

Les variables relatives aux décisions d’asile et à l’IDMC sont utilisées prioritairement sous forme **retardée à T−1** pour sécuriser leur disponibilité au moment du scoring. Le Feature Set V1 privilégie ainsi les informations disponibles à T et dans le passé.


### 7. Feature Set candidat


In [15]:
categorical_features = [
    "coo_id",
    "coa_id",
    "procedure_type",
    "app_type",
    "dec_level",
    "app_pc",
    "origin_region",
    "asylum_region"
]

numeric_features = [
    "year",
    "applied",

    "applied_lag1",
    "applied_lag2",
    "growth_past_1",
    "growth_past_2",
    "absolute_change_past_1",
    "absolute_change_past_2",
    "two_consecutive_increases",
    "applied_mean_3y",
    "applied_std_3y",

    "decisions_recognized_lag1",
    "decisions_other_lag1",
    "decisions_rejected_lag1",
    "decisions_closed_lag1",
    "decisions_total_lag1",
    "refugee_recognition_rate_lag1",
    "protection_rate_lag1",
    "rejection_rate_lag1",
    "has_decisions_data_lag1",

    "idmc_total_origin_lag1",
    "has_idmc_data_lag1"
]

categorical_features = [
    c for c in categorical_features if c in df_fe.columns
]

numeric_features = [
    c for c in numeric_features if c in df_fe.columns
]

final_features = categorical_features + numeric_features

print("Features catégorielles :", len(categorical_features))
print(categorical_features)

print("\nFeatures numériques :", len(numeric_features))
print(numeric_features)

print("\nNombre total de features :", len(final_features))


Features catégorielles : 8
['coo_id', 'coa_id', 'procedure_type', 'app_type', 'dec_level', 'app_pc', 'origin_region', 'asylum_region']

Features numériques : 22
['year', 'applied', 'applied_lag1', 'applied_lag2', 'growth_past_1', 'growth_past_2', 'absolute_change_past_1', 'absolute_change_past_2', 'two_consecutive_increases', 'applied_mean_3y', 'applied_std_3y', 'decisions_recognized_lag1', 'decisions_other_lag1', 'decisions_rejected_lag1', 'decisions_closed_lag1', 'decisions_total_lag1', 'refugee_recognition_rate_lag1', 'protection_rate_lag1', 'rejection_rate_lag1', 'has_decisions_data_lag1', 'idmc_total_origin_lag1', 'has_idmc_data_lag1']

Nombre total de features : 30


### 8. Journal de sélection des colonnes

Chaque colonne est classée avec une décision explicite : `KEEP`, `DROP_TECHNICAL`, `DROP_LEAKAGE`, `DROP_MISSINGNESS`, `DROP_REDUNDANT`, `DROP_T0_UNCERTAIN` ou `DROP_UNUSED`.

L'objectif est de pouvoir répondre à la question: *Parmi toutes les colonnes du dataset initial, lesquelles seront données au modèle, lesquelles seront écartées, et pourquoi ?*


In [17]:
technical_cols = {
    "_row_id",
    "_demand_row_id",
    "decisions_source_rows",
    "solutions_source_rows"
}

future_leakage_cols = {
    "year_t1",
    "applied_t1",
    "growth_t1",
    TARGET
}

solutions_cols = {
    "returned_refugees",
    "resettlement",
    "naturalisation",
    "returned_idps",
    "has_solutions_data"
}

redundant_reference_cols = {
    "coo_name", "coo", "coo_iso",
    "coa_name", "coa", "coa_iso",
    "origin_iso", "origin_iso2",
    "origin_country", "origin_country_fr",
    "origin_region_fr",
    "origin_major_area", "origin_major_area_fr",
    "asylum_iso", "asylum_iso2",
    "asylum_country", "asylum_country_fr",
    "asylum_region_fr",
    "asylum_major_area", "asylum_major_area_fr"
}

current_t_conditional_cols = {
    "decisions_recognized",
    "decisions_other",
    "decisions_rejected",
    "decisions_closed",
    "decisions_total",
    "recognition_rate",
    "rejection_rate",
    "has_decisions_data",
    "idmc_total_origin",
    "has_idmc_data"
}

selection_log = []

for col in df_fe.columns:
    if col in final_features:
        decision = "KEEP"
        reason = "Feature retenue pour le modèle V1"
    elif col in future_leakage_cols:
        decision = "DROP_LEAKAGE"
        reason = "Information future / cible, interdite dans X"
    elif col in technical_cols:
        decision = "DROP_TECHNICAL"
        reason = "Identifiant ou métadonnée technique"
    elif col in solutions_cols:
        decision = "DROP_MISSINGNESS"
        reason = "Variable de solution trop incomplète pour la V1"
    elif col in redundant_reference_cols:
        decision = "DROP_REDUNDANT"
        reason = "Référentiel descriptif redondant"
    elif col in current_t_conditional_cols:
        decision = "DROP_T0_UNCERTAIN"
        reason = "Disponibilité à T non garantie ; version lag1 privilégiée"
    elif col == "has_asylum_application_data":
        decision = "DROP_LOW_VALUE"
        reason = "Indicateur non retenu dans le Feature Set V1"
    else:
        decision = "DROP_UNUSED"
        reason = "Non retenue dans le Feature Set V1"

    selection_log.append({
        "column": col,
        "decision": decision,
        "reason": reason
    })

selection_log = pd.DataFrame(selection_log)

display(
    selection_log.groupby("decision")
    .size()
    .rename("n_columns")
    .sort_values(ascending=False)
    .to_frame()
)

display(selection_log)


,n_columns
decision,
KEEP,30
DROP_UNUSED,21
DROP_REDUNDANT,20
DROP_T0_UNCERTAIN,10
DROP_MISSINGNESS,5
DROP_LEAKAGE,4
DROP_TECHNICAL,4
DROP_LOW_VALUE,1


,column,decision,reason
0,_row_id,DROP_TECHNICAL,Identifiant ou métadonnée technique
1,year,KEEP,Feature retenue pour le modèle V1
2,coo_id,KEEP,Feature retenue pour le modèle V1
3,coo_name,DROP_REDUNDANT,Référentiel descriptif redondant
4,coo,DROP_REDUNDANT,Référentiel descriptif redondant
5,coo_iso,DROP_REDUNDANT,Référentiel descriptif redondant
6,coa_id,KEEP,Feature retenue pour le modèle V1
7,coa_name,DROP_REDUNDANT,Référentiel descriptif redondant
8,coa,DROP_REDUNDANT,Référentiel descriptif redondant
9,coa_iso,DROP_REDUNDANT,Référentiel descriptif redondant


### 9. Création du dataset supervisé

Les lignes sans véritable T+1 ne peuvent pas entraîner un modèle supervisé. Elles sont isolées et exportées pour vérification.


In [18]:
unlabelled_rows = df_fe[df_fe[TARGET].isna()].copy()
df_ml = df_fe[df_fe[TARGET].notna()].copy()

X = df_ml[final_features].copy()
y = df_ml[TARGET].astype(int).copy()

print("Dataset Feature Engineering complet :", df_fe.shape)
print("Dataset supervisé :", df_ml.shape)
print("X :", X.shape)
print("y :", y.shape)
print("Lignes non labellisables conservées pour audit :", len(unlabelled_rows))

target_summary = (
    y.value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)
target_summary["percentage"] = target_summary["count"] / len(y) * 100

display(target_summary)


Dataset Feature Engineering complet : (120597, 95)
Dataset supervisé : (80032, 95)
X : (80032, 30)
y : (80032,)
Lignes non labellisables conservées pour audit : 40565


,count,percentage
hausse_critique_demandes_t1,,
0,48424,60.5058
1,31608,39.4942


### 10. Contrôles anti-leakage et qualité


In [20]:
forbidden_in_X = {
    "year_t1",
    "applied_t1",
    "growth_t1",
    TARGET
}

leakage_found = forbidden_in_X.intersection(X.columns)
assert not leakage_found, f"Leakage détecté dans X : {leakage_found}"

print("Aucun champ futur explicite dans X.")

missing_features = (
    X.isna().mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_pct")
    .to_frame()
)

display(missing_features)

constant_cols = [
    c for c in X.columns
    if X[c].nunique(dropna=False) <= 1
]

print("Colonnes constantes détectées :", constant_cols)


Aucun champ futur explicite dans X.


,missing_pct
idmc_total_origin_lag1,67.1644
absolute_change_past_2,40.2302
two_consecutive_increases,40.2302
growth_past_2,40.2302
applied_std_3y,40.2302
applied_mean_3y,40.2302
applied_lag2,34.9548
rejection_rate_lag1,33.1367
protection_rate_lag1,33.1367
refugee_recognition_rate_lag1,33.1367


Colonnes constantes détectées : []


Avant la modélisation, des contrôles automatiques de qualité du Feature Set est mis en place. Le premier vérifie qu'aucune variable future ou cible n'est présente dans X (features), afin d'éviter le target leakage. Le deuxième quantifie les valeurs manquantes pour chaque feature afin de préparer leur traitement. Le troisième identifie les colonnes constantes (une variable qui contient la même valeur pour toutes les lignes), qui n'apportent aucune information prédictive et peuvent être exclues du modèle.

### 11. Export des livrables


In [1]:
dataset_ml_export = df_ml[final_features + [TARGET]].copy()

dataset_ml_export.to_csv(
    OUTPUT_DIR / "dataset_ml_features.csv",
    index=False
)

selection_log.to_csv(
    OUTPUT_DIR / "feature_selection_log.csv",
    index=False
)

unlabelled_rows.to_csv(
    OUTPUT_DIR / "rows_unlabelled_for_supervised_ml.csv",
    index=False
)

print("Exports créés dans :", OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.iterdir()):
    print("-", p.name)


NameError: name 'df_ml' is not defined

### Synthèse Feature Engineering

Le Feature Set V1 repose sur trois familles principales : **profil du segment**, **dynamique historique** et **contexte retardé**. Les colonnes techniques, descriptives redondantes, trop incomplètes et toutes les informations T+1 sont exclues du dataset ML.

La prochaine phase est la modélisation : **split chronologique → preprocessing train-only → baseline → modèles → comparaison Recall / Precision / F1 / PR-AUC**.
